# Random Forest Regression

In the previous notebook, we built and tuned a Decision Tree Regressor.

Our tuned decision tree achieved a test R² of approximately 0.728.

However, a single decision tree can be sensitive to the particular training data. Small changes in the training data can sometimes produce a different tree structure and therefore different predictions.

Random Forest is an ensemble learning method that addresses this limitation by combining predictions from many decision trees.

In this notebook, we will:

1. Understand why a single decision tree can be unstable.
2. Understand the idea of ensemble learning.
3. Learn how Random Forest combines multiple decision trees.
4. Understand bootstrap sampling.
5. Understand random feature selection.
6. Build a small Random Forest conceptually.
7. Train a `RandomForestRegressor` using Scikit-learn.
8. Compare it with our tuned Decision Tree.
9. Tune the Random Forest using cross-validation.

## Why Use More Than One Decision Tree?

A decision tree learns its structure from the training data.

The training process determines:

- which feature is selected for a split,
- which threshold is selected,
- which samples reach each leaf,
- and therefore the prediction made by each leaf.

Because the tree depends on the training data, changing the data can change the resulting tree.

This means that a single decision tree can have relatively high variance.

Random Forest reduces this problem by training many different decision trees and combining their predictions.

## Ensemble Learning

An **ensemble** is a machine learning model that combines multiple individual models.

Instead of relying on one model:

$$
\text{Prediction} = \text{Model}_1(X)
$$

we can combine several models:

$$
\text{Prediction}
=
\text{combine}(
\text{Model}_1(X),
\text{Model}_2(X),
\dots,
\text{Model}_T(X)
)
$$

Random Forest is an ensemble of decision trees.

For regression, the predictions of the individual trees are averaged:

$$
\hat{y}
=
\frac{1}{T}
\sum_{t=1}^{T}
\hat{y}_t
$$

where:

- $T$ is the number of trees.
- $\hat{y}_t$ is the prediction made by tree $t$.
- $\hat{y}$ is the final Random Forest prediction.

The central idea is therefore:

> Instead of trusting one decision tree, combine the predictions of many trees.

## A Simple Random Forest Example

Suppose five decision trees make the following predictions for one house:

$$
2.0,\ 2.4,\ 2.2,\ 2.6,\ 2.3
$$

The Random Forest averages these predictions:

$$
\hat y =
\frac{2.0+2.4+2.2+2.6+2.3}{5}
=2.3
$$

Therefore, the final prediction of the forest is 2.3.

The individual trees may disagree, but averaging their predictions can produce a more stable prediction.

## Bootstrap Sampling

If we trained every decision tree on exactly the same training data, the trees would tend to learn very similar structures.

Random Forest avoids this by giving each tree a different training dataset.

It creates these datasets using **bootstrap sampling**.

Bootstrap sampling means repeatedly selecting samples from the original training data **with replacement**.

For example, suppose our training dataset contains only:

$$
[A,\ B,\ C,\ D,\ E]
$$

A bootstrap sample could be:

$$
[A,\ C,\ C,\ E,\ A]
$$

Notice that:

- `A` appears twice.
- `C` appears twice.
- `B` and `D` were not selected.
- The bootstrap sample still contains 5 observations.

Another tree might receive:

$$
[B,\ D,\ A,\ D,\ E]
$$

This gives the trees different training data, causing them to learn different structures.

In [3]:
import numpy as np

data = np.array(["A", "B", "C", "D", "E"])

bootstrap_sample = np.random.choice(
    data,
    size = len(data),
    replace=True
)

print("Original data:", data)
print("Bootstrap sample:", bootstrap_sample)

Original data: ['A' 'B' 'C' 'D' 'E']
Bootstrap sample: ['C' 'E' 'B' 'A' 'E']


## Bootstrap Samples in Random Forest

For our California Housing training data, we have 16,512 training observations.

Random Forest can create a bootstrap sample of approximately the same size for each tree by sampling the training observations with replacement.

For example:

$$
\text{Training data}
\rightarrow
\begin{cases}
\text{Bootstrap sample 1} \rightarrow \text{Tree 1}\\
\text{Bootstrap sample 2} \rightarrow \text{Tree 2}\\
\text{Bootstrap sample 3} \rightarrow \text{Tree 3}\\
\vdots\\
\text{Bootstrap sample T} \rightarrow \text{Tree T}
\end{cases}
$$

Because the samples are randomly generated, the trees receive different training data and can therefore learn different structures.

This is one of the sources of randomness in a Random Forest.